# Introduction to Graph Neural Networks (GNNs)

This notebook builds a **Graph Neural Network** from first principles and trains it to
classify scientific papers in the **Cora** citation network. The explanations below
walk through every code fragment, giving the historical background of each idea and
concrete examples of where it is used in practice.

## Why graphs?

Classic deep learning tools are built for data laid out on a regular grid: a
**convolutional network (CNN)** assumes pixels sit on a 2-D lattice, and a
**recurrent network (RNN)** assumes tokens arrive in a 1-D sequence. A great deal of
real-world data is *not* grid-shaped, but instead a set of entities connected by
relationships — a **graph**:

- **Citation networks** — papers (nodes) cite other papers (edges). *Cora*, used here,
  is the "MNIST of graph learning."
- **Social networks** — users connected by friendships (Facebook, Twitter).
- **Molecules** — atoms joined by chemical bonds; GNNs power drug discovery and were
  central to DeepMind's work predicting molecular properties.
- **Recommender systems** — the Pinterest *PinSage* model and many others treat
  users/items as a bipartite graph.
- **Knowledge graphs, road/traffic networks (Google Maps ETA), protein interaction
  graphs (AlphaFold-adjacent tooling), and fraud-detection graphs.**

A GNN generalizes the convolution idea from grids to arbitrary graphs: each node
updates its representation by **aggregating messages from its neighbours**. The lineage
runs from Gori et al. (2005) and Scarselli et al. (2009), who coined "graph neural
network," through the spectral formulations of Bruna et al. (2014) and Defferrard et al.
(2016), to the model we reimplement here: Kipf & Welling's **Graph Convolutional
Network (GCN)**, "Semi-Supervised Classification with Graph Convolutional Networks"
(ICLR 2017), which made GNNs simple, fast, and popular.

## Acknowledgements

This notebook follows the live coding demo from Petar Veličković's talk
**[Intro to graph neural networks (ML Tech Talks)](https://www.youtube.com/watch?v=8owQBFAHw7E)** —
huge thanks to him for such a clear introduction to GNNs. The model reimplements
Kipf & Welling's GCN (*"Semi-Supervised Classification with Graph Convolutional
Networks"*, ICLR 2017), using the [Spektral](https://graphneural.network/) library
by Daniele Grattarola.

## 1. Installing Spektral

```python
!pip install spektral
```

The leading `!` tells Jupyter/IPython to run the rest of the line as a **shell command**
rather than as Python. `pip` is Python's package installer (the name is a recursive
acronym, "Pip Installs Packages"); it fetches the library and its dependencies from
**PyPI**, the Python Package Index.

**What is Spektral?** [Spektral](https://graphneural.network/) is an open-source library
for building GNNs on top of **Keras** and **TensorFlow 2**, created by Daniele
Grattarola and Cesare Alippi (paper: *"Graph Neural Networks in TensorFlow and Keras
with Spektral"*, 2020). It is the TensorFlow-world counterpart to **PyTorch Geometric**
(PyG) and the **Deep Graph Library** (DGL) in the PyTorch world. Here we mainly use it
for one convenience: downloading and preparing the Cora dataset.

**Historical context.** Before package managers like `pip` (2008) and repositories like
PyPI, sharing scientific Python code meant manually distributing source archives. The
`!pip install` one-liner inside a notebook is what makes cloud environments such as
**Google Colab** and Kaggle Kernels "just work" — you provision dependencies in the
same document that contains your experiment. The output log shows `pip` resolving the
dependency tree (TensorFlow, NumPy, SciPy, NetworkX, scikit-learn, …) and even
*downgrading* `protobuf` to satisfy TensorFlow's version constraints — a everyday
example of **dependency resolution**.

In [1]:
!pip install spektral

     -------------------------------------- 140.1/140.1 kB 2.1 MB/s eta 0:00:00
     ---------------------------------------- 1.5/1.5 MB 6.8 MB/s eta 0:00:00
     -------------------------------------- 895.9/895.9 kB 7.1 MB/s eta 0:00:00
     ---------------------------------------- 23.2/23.2 MB 2.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.1
    Uninstalling protobuf-3.20.1:
      Successfully uninstalled protobuf-3.20.1


## 2. Verifying the installation

```python
!pip show spektral
```

`pip show` prints the **metadata** of an already-installed package: its version, author,
license, home page, and — crucially — its `Requires` and `Required-by` fields. Reading
those two lines tells you what Spektral depends on and what depends on Spektral.

**Why bother?** Reproducibility. Machine-learning results are notoriously sensitive to
library versions (an API change in TensorFlow or a different default in scikit-learn can
move your accuracy). Recording the exact version — here **Spektral 1.2.0** — is a small
but real part of making an experiment repeatable. In production this same instinct scales
up to lockfiles (`requirements.txt`, `pip freeze`, `poetry.lock`, Conda environment
files) and containerization with Docker.

In [2]:
!pip show spektral

Name: spektral
Version: 1.2.0
Summary: Graph Neural Networks with Keras and Tensorflow 2.
Home-page: https://github.com/danielegrattarola/spektral
Author: Daniele Grattarola
Author-email: daniele.grattarola@gmail.com
License: MIT
Location: c:\users\mohds\anaconda3\lib\site-packages
Requires: joblib, lxml, networkx, numpy, pandas, requests, scikit-learn, scipy, tensorflow, tqdm
Required-by: 


## 3. Importing the core libraries

```python
import numpy as np
import tensorflow as tf
import spektral
```

Three imports, three pillars of the modern ML stack:

- **NumPy** (`np`) — the foundational array library for numerical Python (originating
  from Numeric/Numarray, unified as NumPy in 2006). Everything numeric — the feature
  matrix, the adjacency matrix, the train/val/test masks — is ultimately a NumPy array.
  The `import numpy as np` alias is a near-universal community convention.
- **TensorFlow** (`tf`) — Google's deep-learning framework, open-sourced in 2015. This
  notebook uses **TensorFlow 2.x**, whose defining feature is **eager execution**:
  operations run immediately (like normal Python) instead of building a static graph
  first, and gradients are recorded with `tf.GradientTape`. We rely on `tf` for dense
  layers, matrix multiplication, automatic differentiation, and the Adam optimizer.
- **Spektral** — the GNN library from step 1.

Aliasing (`as np`, `as tf`) simply shortens the names you type thousands of times.

In [3]:
import numpy as np
import tensorflow as tf
import spektral

## 4. Loading the Cora citation network

```python
cora_dataset = spektral.datasets.citation.Citation(name='cora')
test_mask  = cora_dataset.mask_te
train_mask = cora_dataset.mask_tr
val_mask   = cora_dataset.mask_va
graph = cora_dataset.graphs[0]
features = graph.x
adj      = graph.a
labels   = graph.y
```

### The Cora dataset

**Cora** is the canonical benchmark for node classification — the "MNIST of graph
learning." It was assembled in the late 1990s (McCallum et al.) and the standardized
"planetoid" split used here was popularized by Yang et al. (2016) and Kipf & Welling
(2017). It contains **2,708 machine-learning papers** connected by **citation links**.
The printed shapes tell the whole story:

- `features.shape == (2708, 1433)` — each of the 2,708 papers is described by a
  **1,433-dimensional bag-of-words** vector: a 0/1 flag for whether each word in a fixed
  vocabulary appears in the paper. This is the classic pre-embeddings text
  representation.
- `adj.shape == (2708, 2708)` — the **adjacency matrix** `A`. Entry `A[i, j] = 1` means
  paper *i* cites (or is cited by) paper *j*. It is stored **sparsely** (as a SciPy
  sparse matrix) because almost all of its ~7.3 million entries are zero — real networks
  are sparse.
- `labels.shape == (2708, 7)` — each paper belongs to one of **7 topics** (e.g.
  *Neural Networks*, *Reinforcement Learning*, *Probabilistic Methods*), stored as a
  **one-hot** vector.

### The masks and *transductive, semi-supervised* learning

`np.sum(train_mask) == 140`, `val == 500`, `test == 1000`. The masks are boolean vectors
that select which nodes count for training, validation, and testing. Notice we train on
only **140 labelled nodes (about 5%)** yet evaluate on 1,000 — this is
**semi-supervised node classification**. The key insight of GCNs is that even the
*unlabelled* nodes contribute, because the graph structure lets label information
"diffuse" along citation edges.

This setup is **transductive**: the model sees the *entire graph* (all nodes and all
edges, including test nodes' features) during training and only the test nodes' *labels*
are hidden. That mirrors real applications — a citation graph or social network is fully
observed; what you lack is labels for most of it.

The `FutureWarning`/`SparseEfficiencyWarning` in the output are harmless deprecation
notes from NetworkX and SciPy about how the sparse matrix is built.

In [7]:
cora_dataset = spektral.datasets.citation.Citation(name='cora')
test_mask = cora_dataset.mask_te
train_mask = cora_dataset.mask_tr
val_mask = cora_dataset.mask_va
graph = cora_dataset.graphs[0]
features = graph.x
adj = graph.a
labels = graph.y

print(features.shape)
print(adj.shape)
print(labels.shape)

print(np.sum(train_mask))
print(np.sum(val_mask))
print(np.sum(test_mask))

(2708, 1433)
(2708, 2708)
(2708, 7)
140
500
1000


C:\Users\MohdS\anaconda3\lib\site-packages\spektral\datasets\citation.py:108: FutureWarning: adjacency_matrix will return a scipy.sparse array instead of a matrix in Networkx 3.0.
  a = nx.adjacency_matrix(nx.from_dict_of_lists(graph))  # CSR
C:\Users\MohdS\anaconda3\lib\site-packages\scipy\sparse\_index.py:146: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


## 5. Masked loss and masked accuracy

```python
def masked_softmax_cross_entropy(logits, labels, mask):
    loss = tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=labels)
    mask = tf.cast(mask, dtype=tf.float32)
    mask /= tf.reduce_mean(mask)
    loss *= mask
    return tf.reduce_mean(loss)

def masked_accuracy(logits, labels, mask):
    correct_prediction = tf.equal(tf.argmax(logits, 1), tf.argmax(labels, 1))
    accuracy_all = tf.cast(correct_prediction, tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    mask /= tf.reduce_mean(mask)
    accuracy_all *= mask
    return tf.reduce_mean(accuracy_all)
```

Because the GNN always outputs a prediction for **all 2,708 nodes**, but we may only
train (or test) on a subset, we need loss/metric functions that **ignore the nodes
outside the current mask**. These two helpers come almost verbatim from Kipf & Welling's
original GCN reference implementation.

**How the masking trick works** (the elegant part):

1. `softmax_cross_entropy_with_logits` computes the standard classification loss for
   *every* node. **Softmax** turns the raw scores (`logits`) into a probability
   distribution over the 7 classes; **cross-entropy** measures how far that distribution
   is from the one-hot truth. Passing raw logits (rather than pre-softmaxed
   probabilities) is done for numerical stability — a well-known deep-learning gotcha.
2. `mask /= tf.reduce_mean(mask)` **re-normalizes** the mask. If, say, 140 of 2,708
   entries are 1, the mean is `140/2708`; dividing by it rescales the ones so that the
   surviving entries average to 1. This means the subsequent `tf.reduce_mean` over
   **all** nodes yields exactly the mean over the *masked* nodes — a clean way to average
   over a subset without changing tensor shapes.
3. `loss *= mask` zeroes out the contribution of unmasked nodes; the final
   `reduce_mean` gives the average loss on just the training nodes.

`masked_accuracy` applies the identical trick to a 0/1 "was the prediction correct"
signal: `tf.argmax(logits, 1)` picks the highest-scoring class per node and compares it
to the true class. This masked-metric pattern is standard whenever you evaluate on
irregular subsets — sequence models use the same idea to ignore padding tokens.

In [8]:
def masked_softmax_cross_entropy(logits, labels, mask):
    loss = tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=labels)
    mask = tf.cast(mask, dtype=tf.float32)
    mask /= tf.reduce_mean(mask)
    loss *= mask
    return tf.reduce_mean(loss)

def masked_accuracy(logits, labels, mask):
    correct_prediction = tf.equal(tf.argmax(logits, 1), tf.argmax(labels, 1))
    accuracy_all = tf.cast(correct_prediction, tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    mask /= tf.reduce_mean(mask)
    accuracy_all *= mask
    return tf.reduce_mean(accuracy_all)

## 6. The heart of it — one GNN layer

```python
def gnn(fts, adj, transform, activation):
    seq_fts = transform(fts)      # 1. transform each node's features
    adj = adj.todense()           #    (densify the sparse adjacency)
    ret_fts = tf.matmul(adj, seq_fts)  # 2. aggregate over neighbours
    return activation(ret_fts)    # 3. non-linearity
```

This tiny function *is* a graph convolution, and it captures the single most important
idea in the whole field: **message passing / neighbourhood aggregation**.

Read it as three steps applied to the node-feature matrix `H` (shape `[N, features]`):

1. **Transform** — `transform(fts)` applies a learnable linear layer (a
   `tf.keras.layers.Dense`), i.e. `H · W`. Every node's feature vector is projected
   through the *same shared weights* `W`. Sharing weights across all nodes is the graph
   analogue of a CNN sliding one filter across every pixel — it is what makes the model
   generalize and keeps the parameter count independent of graph size.
2. **Aggregate** — `tf.matmul(adj, seq_fts)` computes `A · (H·W)`. Multiplying by the
   adjacency matrix `A` **sums each node's neighbours' transformed features**: row *i* of
   the result is the sum over all *j* with `A[i,j]=1`. This is exactly "gather messages
   from your neighbours and add them up." Stacking two such layers lets information travel
   **two hops** across the graph (a paper is influenced by the papers it cites *and* the
   papers those cite).
3. **Activate** — `activation(...)` applies a non-linearity (here ReLU for the hidden
   layer, identity for the output), just as in an ordinary neural net.

### Historical context and how this relates to the "real" GCN

The propagation rule of Kipf & Welling's GCN is
`H⁽ˡ⁺¹⁾ = σ( Â · H⁽ˡ⁾ · W⁽ˡ⁾ )`, where `Â` is the adjacency matrix **with self-loops
added and symmetrically normalized** (`Â = D^{-1/2}(A+I)D^{-1/2}`). This notebook uses a
deliberately **simplified** version: plain `A`, without self-loops or degree
normalization. That simplification is pedagogically perfect — it exposes the bare
`transform → aggregate → activate` skeleton — but it also explains the modest accuracy
(~75%; the full normalized GCN reaches ~81% on Cora). Without normalization, high-degree
nodes produce large-magnitude sums and the scale of features grows layer to layer; the
`D^{-1/2}…D^{-1/2}` term fixes that, and the self-loop `+I` lets a node keep its own
features, not just its neighbours'.

This aggregation view unifies the modern GNN zoo: **GraphSAGE** (Hamilton et al., 2017)
swaps the sum for mean/max/LSTM aggregators and samples neighbours to scale to billions
of edges; **GAT** (Veličković et al., 2018) replaces the fixed `A` weights with
*learned attention*; **GIN** (Xu et al., 2019) chooses an aggregator provably as powerful
as the Weisfeiler–Lehman graph-isomorphism test.

> **Performance note:** `adj.todense()` converts the sparse adjacency into a full dense
> matrix on every call. It keeps the code short and readable, but for large graphs it is
> wasteful — real implementations use **sparse matrix multiplication** (`tf.sparse` /
> `torch.sparse`) so cost scales with the number of edges, not `N²`.

In [12]:
def gnn(fts, adj, transform, activation):
    seq_fts = transform(fts)
    adj = adj.todense()
    ret_fts = tf.matmul(adj, seq_fts)
    return activation(ret_fts)

## 7. Assembling and training the model

```python
def train_cora(fts, adj, gnn_fn, units, epochs, lr):
    lyr_1 = tf.keras.layers.Dense(units)   # hidden layer: -> `units` dims
    lyr_2 = tf.keras.layers.Dense(7)       # output layer: -> 7 classes

    def cora_gnn(fts, adj):
        hidden = gnn_fn(fts, adj, lyr_1, tf.nn.relu)      # GNN layer 1 + ReLU
        logits = gnn_fn(hidden, adj, lyr_2, tf.identity)  # GNN layer 2 (linear)
        return logits

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    best_accuracy = 0.0
    for ep in range(epochs + 1):
        with tf.GradientTape() as t:
            logits = cora_gnn(fts, adj)
            loss = masked_softmax_cross_entropy(logits, labels, train_mask)

        variables = t.watched_variables()
        grads = t.gradient(loss, variables)
        optimizer.apply_gradients(zip(grads, variables))

        logits = cora_gnn(fts, adj)
        val_accuracy  = masked_accuracy(logits, labels, val_mask)
        test_accuracy = masked_accuracy(logits, labels, test_mask)

        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            print('Epoch', ep, '| Training loss:', loss.numpy(), 'Val accuracy:',
                  val_accuracy.numpy(), '|Test accuracy:', test_accuracy.numpy())
```

This puts the pieces together into a complete **two-layer GCN** and trains it.

### Building the network

- `lyr_1 = Dense(units)` and `lyr_2 = Dense(7)` create the two learnable weight matrices
  `W⁽¹⁾` and `W⁽²⁾`. They are made **once, outside the loop**, so the same parameters are
  reused and updated every epoch (a common bug is recreating layers inside the loop,
  which resets the weights).
- `cora_gnn` stacks two of our `gnn` layers. The first uses **ReLU** (`tf.nn.relu`,
  `max(0, x)` — the activation that, since Krizhevsky et al.'s AlexNet in 2012, made deep
  nets trainable). The second uses **identity** (`tf.identity`) because its outputs are
  *logits* fed straight into softmax-cross-entropy. **Two layers = a two-hop receptive
  field**, the standard depth for GCNs — going much deeper causes *over-smoothing*, where
  all node representations collapse toward the same vector.

### The training loop

- **`tf.GradientTape`** is TensorFlow 2's **automatic differentiation** mechanism. Inside
  the `with` block it *records* every operation touching trainable variables; afterwards
  `t.gradient(loss, variables)` replays that tape backwards to compute
  ∂loss/∂parameters — this is **backpropagation**, made concrete.
- `t.watched_variables()` returns the parameters the tape saw (the two dense layers'
  weights and biases, created lazily on first call).
- **`optimizer.apply_gradients`** performs the update step. **Adam** (Kingma & Ba, 2015)
  is an adaptive-learning-rate optimizer that keeps per-parameter running averages of the
  gradient and its square; it is the default first choice across deep learning.
- **Full-batch gradient descent:** because Cora is small, every epoch uses *all* nodes at
  once — there is no mini-batching here.

### Early-stopping-style model selection

Each epoch, accuracy is measured on the **validation** set, and a line is printed only
when validation accuracy hits a new best. Watching **val** rather than **train** accuracy
to judge progress is the guard against **overfitting**; the parallel **test** number is
reported alongside purely for our curiosity — in a rigorous protocol you would keep the
final test evaluation strictly for the single selected model.

In [13]:
def train_cora(fts, adj, gnn_fn, units, epochs, lr):
    lyr_1 = tf.keras.layers.Dense(units)
    lyr_2 = tf.keras.layers.Dense(7)
    
    def cora_gnn(fts, adj):
        hidden = gnn_fn(fts, adj, lyr_1, tf.nn.relu)
        logits = gnn_fn(hidden, adj, lyr_2, tf.identity)
        return logits
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    
    best_accuracy = 0.0
    for ep in range(epochs + 1):
        with tf.GradientTape() as t:
            logits = cora_gnn(fts, adj)
            loss = masked_softmax_cross_entropy(logits, labels, train_mask)
            
        variables = t.watched_variables()
        grads = t.gradient(loss, variables)
        optimizer.apply_gradients(zip(grads, variables))
        
        logits = cora_gnn(fts, adj)
        val_accuracy = masked_accuracy(logits, labels, val_mask)
        test_accuracy = masked_accuracy(logits, labels, test_mask)
        
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            print('Epoch', ep, '| Training loss:', loss.numpy(), 'Val accuracy:',
                  val_accuracy.numpy(), '|Test accuracy:', test_accuracy.numpy())

## 8. Running the training

```python
train_cora(features, adj, gnn, 32, 200, 0.01)
```

Finally we call the trainer with concrete **hyperparameters**:

- `features`, `adj` — Cora's feature and adjacency matrices from step 4.
- `gnn` — we pass our layer function itself as an argument (Python treats functions as
  first-class objects). This dependency-injection style means you could swap in a fancier
  layer — a normalized GCN, a GAT with attention — **without touching the training loop**.
- `units = 32` — the hidden layer has 32 units; 16–64 is typical for Cora.
- `epochs = 200` — the standard number of passes used in the GCN paper.
- `lr = 0.01` — Adam's learning rate, also the paper's value.

### Reading the output

The log shows validation accuracy climbing from **~28% at epoch 0** (essentially random
— 1/7 ≈ 14%, plus a lucky start) to **~75% by epoch 10**, with test accuracy tracking it
closely. In a handful of epochs, training on only **140 labelled papers**, the model
correctly classifies three-quarters of a **thousand held-out** papers. That is the
payoff of exploiting graph structure: the citation edges let scarce label information
propagate to unlabelled nodes.

The gap to the full GCN's ~81% is exactly the price of the simplifications discussed in
step 6 (no self-loops, no symmetric degree normalization). Adding those — the natural
next exercise — closes most of it.

### Where this goes next

The same `transform → aggregate → activate` recipe, scaled up and refined, underlies
production systems: **Pinterest's PinSage** for recommendations, **Google Maps** ETA
prediction via GNNs on road graphs, molecular property prediction in drug discovery, and
fraud detection over transaction graphs. Cora is where nearly everyone starts.

In [14]:
train_cora(features, adj, gnn, 32, 200, 0.01)

Epoch 0 | Training loss: 3.465349 Val accuracy: 0.284 |Test accuracy: 0.298
Epoch 1 | Training loss: 5.3994603 Val accuracy: 0.52599996 |Test accuracy: 0.545
Epoch 2 | Training loss: 2.0442512 Val accuracy: 0.626 |Test accuracy: 0.606
Epoch 4 | Training loss: 0.9207304 Val accuracy: 0.64 |Test accuracy: 0.624
Epoch 5 | Training loss: 0.82862926 Val accuracy: 0.666 |Test accuracy: 0.66499996
Epoch 6 | Training loss: 0.6962547 Val accuracy: 0.69600004 |Test accuracy: 0.695
Epoch 9 | Training loss: 0.5267703 Val accuracy: 0.74 |Test accuracy: 0.74399996
Epoch 10 | Training loss: 0.41576472 Val accuracy: 0.75200003 |Test accuracy: 0.75200003
